# Amplitude amplification for 5-Queens

Execute a reversible constraint oracle on 15 board qubits and four counter qubits. Column-range, column-collision and diagonal predicates determine the phase; no solution list enters oracle construction. Independent classical solvers validate returned boards and provide the known count $M=10$ for the 44-iteration benchmark.

The circuit implements $Q=(2|\psi\rangle\langle\psi|-I)(I-2\Pi_{\mathrm{good}})$ and checks $P_k=\sin^2((2k+1)\arcsin\sqrt{10/32768})$.


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'amplification.py').is_file())
sys.path.insert(0, str(ROOT))
from amplification import queens_constraint_oracle
from experiments import run_experiments, plot_report
oracle = queens_constraint_oracle(5)  # only board size is supplied
print({'board_qubits': 15, 'counter_qubits': oracle.num_qubits-15})
report = run_experiments()


## Execution and returned solutions

Count violations, flip phase at zero, then reverse the computation. Four counter bits cover all 15 possible predicates without overflow. All ten boards below come from the seeded constraint-circuit measurements and pass separate classical validation. Row 0 occupies the most significant three-bit group.


In [ ]:
result = report['queens5']['constraint_run']
print(json.dumps({k: result[k] for k in (
    'iterations','exact_success','sampled_success','valid_shots','invalid_shots',
    'binomial_95_interval','state_error_l2','workspace_leakage')}, indent=2))
print('Recovered boards (row -> column):')
for board in report['queens5']['observed_valid_boards']:
    print(board)


In [ ]:
print('Logical resources (u/CX basis, no routing):')
for name in ('queens5_constraint_oracle','queens5_lookup_oracle','queens5_modular_run'):
    print(name, report['resources'][name])
print('Classical backtracking column attempts:', report['queens5']['classical']['backtracking_column_attempts'])
print('Supporting cases:', {
    'uniform': report['uniform']['exact_success'],
    'entangled_nonuniform': report['nonuniform']['exact_success'],
    'four_queens': report['queens4']['constraint_run']['exact_success']})


## Behavior and validation

Tests probe every binary board assignment, complete complex amplitudes, counter uncomputation and iteration overshooting. The lookup oracle is a comparison that already knows the answers. Costs distinguish oracle calls from synthesized gates. Results use ideal simulation and known success probability; the [README](README.md) records scope and reproduction.


In [ ]:
from IPython.display import SVG, display
figure = ROOT/'artifacts'/'notebook'/'amplification.svg'
plot_report(report, figure)
display(SVG(filename=str(figure)))
